# Mechanism algorithms tutorial

## Running the provided fishery benchmark with quota, incentives, social influence, and composition

This notebook walks through the mechanism abstraction of the bilevel fishery
project: what a *mechanism* is mathematically, how each supplied algorithm
(quota, subsidy, threshold penalty, social observation) maps onto the code, how
mechanisms are composed, and how the regulated fishery environment applies them
during one step.

Every code cell executes in well under a minute on a laptop. The notebook does
**not** train any policy: the full bilevel run (Evolution Strategies over the
mechanism parameters, APPO for the fishers) is launched from the command line
with `python -m examples.bilevel_fishery.debug`, see section 10.

Run it from the repository root (`uv run jupyter lab tutorials/...`) so that the
`core` and `examples` packages import; the first code cell adds the parent
directory to `sys.path` as a fallback.

**Reading order.** Read this notebook first, then
`custom_benchmark_creation.ipynb`, which reuses the mechanism API introduced
here to build a new benchmark and a new mechanism.

In [ ]:
import os
import sys

# ``core`` is an installed package but ``examples`` is not: when the kernel
# starts inside ``tutorials/``, import both from the repository root.
try:
    import examples.bilevel_fishery  # noqa: F401
except ImportError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import matplotlib.pyplot as plt
import numpy as np

EPS = 1e-8
np.set_printoptions(precision=4, suppress=True)

# 1. Reinforcement learning before mechanisms

A Markov decision process can be written as:

$$
\mathcal E
=
(\mathcal S,\mathcal A,P,R,\gamma).
$$

At time $t$:

1. the environment is in state $s_t$;
2. the agent receives observation $o_t$;
3. the policy samples:

$$
a_t \sim \pi_\phi(\cdot\mid o_t);
$$

4. the environment transitions:

$$
s_{t+1}\sim P(\cdot\mid s_t,a_t);
$$

5. the agent receives reward $r_t$.

A Bellman-style value equation is:

$$
V^{\pi_\phi}(s_t)
=
\mathbb E
\left[
r_t+\gamma V^{\pi_\phi}(s_{t+1})
\right].
$$

The learned policy depends on the experience generated by this loop.
Therefore, changing observations, actions, or rewards changes the policy
optimization problem.

# 2. Mechanism intervention

We represent a mechanism as:

$$
\mathcal M_\theta
=
\left(
\mathcal M_\theta^O,
\mathcal M_\theta^A,
\mathcal M_\theta^R
\right).
$$

Observation shaping:

$$
o_t^*=\mathcal M_\theta^O(s_t,o_t).
$$

The policy acts on:

$$
a_t\sim \pi_\phi(\cdot\mid o_t^*).
$$

Action shaping:

$$
a_t^*=\mathcal M_\theta^A(s_t,a_t).
$$

The environment transition receives $a_t^*$:

$$
s_{t+1}\sim P(\cdot\mid s_t,a_t^*).
$$

Reward shaping:

$$
r_t^*
=
\mathcal M_\theta^R(
r_t,
s_t,
a_t^*,
s_{t+1}
).
$$

The learner therefore optimizes:

$$
V^{\pi_\phi,\theta}(s_t)
=
\mathbb E
\left[
r_t^*+\gamma V^{\pi_\phi,\theta}(s_{t+1})
\right].
$$

This separation lets a benchmark describe ecology/physics while a mechanism
describes regulation.

## 2.1 How this maps to code

`core.mechanism.base.Mechanism` is an abstract base class with two faces.

The **channel face** is the triple above. Each channel is a method taking a
per-agent dictionary and returning a per-agent dictionary; the defaults are
identities, so a concrete mechanism overrides only the channels it uses:

```python
mechanism.action(action_dict, **context)
mechanism.reward(reward_dict, **context)
mechanism.observation(observation_dict, **context)
```

The **optimizer face** makes the mechanism a point in a search space:
`dimension`, `encode()` (free parameters as a vector in $[0,1]^d$),
`decode(x)` (a new mechanism with those parameters), `clip()`, `param_names()`
and `to_vector()` (the full parameter vector shown to agents, which may include
fixed parameters). Fixed rules have `dimension == 0`.

A mechanism may require benchmark-specific runtime values. Those are injected
through **bindings**: callables `env -> value` stored on the mechanism.
`Mechanism.resolve(env)` evaluates them and the environment passes the result
as keyword arguments to the channel methods. `resolve` is therefore a helper
that feeds the channels, not a fourth channel: it never transforms actions,
rewards or observations.

```python
bindings={
    "resource_level": lambda env: env.S_t["fish"] / env.K,
}
# Mechanism.resolve(env) -> {"resource_level": 0.73}
```

This lets the same quota algorithm operate on fish biomass, water level, or
another normalized resource without hard-coding the benchmark.

In [ ]:
import inspect

from core.mechanism.base import Mechanism

abstract = sorted(Mechanism.__abstractmethods__)
concrete = {
    name
    for name, member in inspect.getmembers(Mechanism, inspect.isfunction)
    if not getattr(member, "__isabstractmethod__", False) and not name.startswith("_")
}
channels = sorted(concrete & {"action", "reward", "observation"})
print("Abstract (optimizer face):", abstract)
print("Channel face (identity by default):", channels)
print("Bindings resolver (not a channel):", sorted(concrete - set(channels)))


# 3. Fishery benchmark mathematics

Let:

$$
B_t=\text{fish biomass}.
$$

The Pella-Tomlinson growth model is:

$$
G(B_t)
=
\frac{r}{p}
B_t
\left[
1-
\left(\frac{B_t}{K}\right)^p
\right],
$$

where:

- $r$: intrinsic growth rate;
- $K$: carrying capacity;
- $p$: shape parameter;
- $p=1$: Schaefer/logistic case.

With stochastic growth $\epsilon_t$, restoration $I_t$, and realized
harvest $H_t$:

$$
B_{t+1}
=
\operatorname{clip}
\left(
B_t+G(B_t)+\epsilon_t+I_t-H_t
\right).
$$

The benchmark uses a two-component agent action:

$$
a_{i,t}
=
\begin{bmatrix}
h_{i,t}\\
e_{i,t}
\end{bmatrix},
$$

with:

- $h_{i,t}$: harvest fraction (component 0) of the agent's maximal request
  $H^{\max}_{i,t} = m\,F_{\mathrm{msy}}\,B_t/N$;
- $e_{i,t}$: restoration effort (component 1), converted to biomass through
  `ecology_cfg["restoration_effectiveness"]` (default `0.0`, i.e. restoration is
  inert unless the benchmark enables it).

The implementation lives in `examples/bilevel_fishery/regulated_env.py`.

# 4. Quota mechanism

The quota is an **action-shaping mechanism**.

It modifies the harvest action before the benchmark transition.

## 4.1 Mathematical formulation

Define normalized biomass:

$$
b_t=\frac{B_t}{K}\in[0,1].
$$

Let the quota threshold be:

$$
q\in[0,1].
$$

With transition width $w_q$:

$$
L=\sigma\left(\frac{0-q}{w_q}\right),
$$

$$
U=\sigma\left(\frac{1-q}{w_q}\right),
$$

$$
C_t=\sigma\left(\frac{b_t-q}{w_q}\right).
$$

The allowed action fraction is:

$$
\alpha_t
=
\frac{C_t-L}{U-L}.
$$

Interpretation:

- resource far below threshold -> $\alpha_t\approx 0$;
- resource far above threshold -> $\alpha_t\approx 1$;
- near threshold -> smooth transition.

For requested harvest fraction $h_{i,t}$:

$$
h_{i,t}^{*}
=
h_{i,t}
-
\operatorname{smooth}_{+}
\left(
h_{i,t}-\alpha_t;
w_u
\right).
$$

This approximates:

$$
h_{i,t}^{*}\approx \min(h_{i,t},\alpha_t).
$$

Therefore:

$$
a_{i,t}^{*}
=
\begin{bmatrix}
h_{i,t}^{*}\\
e_{i,t}
\end{bmatrix}.
$$

## 4.2 Quota configuration variables

```text
fixed_quota
    q in the equations above. The only optimized parameter (dimension 1).

action_component
    Which component is regulated.
    Fishery: 0 = harvest.

quota_transition_width
    w_q. Smoothness around the quota threshold (default 0.03).

usage_transition_width
    w_u. Smoothness of the action cap (default 0.005).

bindings["resource_level"]
    Runtime normalized resource state b_t.
```

## 4.3 Constructing a quota mechanism

The constructor validates the bindings it needs (`resource_level`) and the
parameter ranges. The mechanism is a frozen dataclass: `decode()` returns a new
instance instead of mutating this one.

In [ ]:
from core.mechanism.algorithms.quota import QuotaMechanism

quota = QuotaMechanism(
    fixed_quota=0.56224,
    action_component=0,
    quota_transition_width=0.03,
    usage_transition_width=0.005,
    bindings={
        "resource_level": lambda env: env.S_t["fish"] / max(env.K, EPS),
    },
)

print("dimension  :", quota.dimension)
print("param_names:", quota.param_names())
print("encode()   :", quota.encode())
print("to_vector():", quota.to_vector())
print("decode(0.3):", quota.decode(np.array([0.3])))

The allowed fraction $\alpha_t$ only depends on the resource level, so we can
plot it directly with `allowed_fraction()` for a few thresholds $q$.

In [ ]:
levels = np.linspace(0.0, 1.0, 400)

fig, ax = plt.subplots(figsize=(6, 3.5))
for q in (0.2, 0.56224, 0.8):
    q_mech = quota.decode(np.array([q]))
    ax.plot(levels, [q_mech.allowed_fraction(b) for b in levels], label=f"q = {q}")
ax.set_xlabel("normalized resource level  b = B / K")
ax.set_ylabel("allowed fraction  alpha")
ax.set_title("Quota: allowed harvest fraction")
ax.legend()
plt.show()

Applying the action channel by hand: the environment would pass
`resource_level=` from the binding; here we pass it ourselves. Requests below
$\alpha_t$ are untouched, requests above are capped, and the restoration
component (index 1) is left alone.

In [ ]:
requested = {
    "u0": np.array([0.9, 0.4], dtype=np.float32),
    "u1": np.array([0.2, 0.7], dtype=np.float32),
}
b_t = 0.5
alpha = quota.allowed_fraction(b_t)
delivered = quota.action(requested, resource_level=b_t)

print(f"allowed fraction at b={b_t}: {alpha:.4f}")
for agent_id in requested:
    print(
        agent_id, "requested", requested[agent_id], "-> delivered", delivered[agent_id]
    )

The quota also uses the **observation channel**: it appends $\alpha_t$ to each
agent's observation so the policy can see how binding the quota is. The value is
cached by `action()` within the step (and recomputed from `resource_level` at
reset, before any action has been taken).

In [ ]:
base_obs = {"u0": np.array([0.5, 0.01]), "u1": np.array([0.5, 0.01])}
print(quota.observation(base_obs, resource_level=b_t))

The same algorithm can be reused in a water benchmark:

```python
bindings={
    "resource_level": lambda env: (
        env.S_t["reservoir"] / env.reservoir_capacity
    )
}
```

# 5. Learned incentive / restoration subsidy

The subsidy is a **reward-shaping mechanism**.

The benchmark determines the ecological effect of restoration.
The mechanism determines the incentive attached to restoration.

## 5.1 General learned incentive

A learned incentive designer can be represented as:

$$
u_t=\mu_\eta(s_t,a_t),
$$

and:

$$
R_\eta^{(i)}(s_t,a_t)
=
R^{(i)}
\left(
s_t,
a_t,
\mu_\eta(s_t,a_t)
\right).
$$

In this codebase, the subsidy mechanism is simpler: the outer optimizer may
optimize a subsidy coefficient. It is not itself a neural meta-gradient
incentive network.

## 5.2 Fish habitat restoration incentive

Let:

- $r_{i,t}$: base reward;
- $e_{i,t}$: restoration effort;
- $c$: cost coefficient;
- $\sigma_\theta$: subsidy.

Then:

$$
r_{i,t}^{*}
=
r_{i,t}
-
c e_{i,t}^{2}
+
\sigma_\theta e_{i,t}.
$$

The quadratic cost discourages simply saturating restoration at its maximum.

## 5.3 Subsidy configuration

```text
subsidy
    sigma_theta: linear incentive per unit restoration effort, in
    [0, MAX_SUBSIDY] with MAX_SUBSIDY = 0.5. The optimized parameter
    (dimension 1); encode() normalizes it by MAX_SUBSIDY and param_names()
    reports it as "restoration_subsidy".

cost
    c: quadratic restoration cost in [0, 1]. Fixed (not optimized).

action_component
    Fishery: 1 = restoration effort.
```

The reward channel needs the delivered actions; the environment supplies them
as the keyword argument `action_after`.

In [ ]:
from core.mechanism.algorithms.subsidy import MAX_SUBSIDY, SubsidyMechanism

subsidy = SubsidyMechanism(subsidy=0.10, cost=0.05, action_component=1)

print("dimension  :", subsidy.dimension)
print("param_names:", subsidy.param_names())
print(
    "encode()   :",
    subsidy.encode(),
    f"(= subsidy / MAX_SUBSIDY, MAX_SUBSIDY={MAX_SUBSIDY})",
)
print("decode(1.0):", subsidy.decode(np.array([1.0])))

base_rewards = {"u0": 0.5, "u1": 0.5}
shaped = subsidy.reward(base_rewards, action_after=delivered)
for agent_id, action in delivered.items():
    effort = action[1]
    expected = 0.5 + 0.10 * effort - 0.05 * effort**2
    print(
        f"{agent_id}: effort={effort:.3f}  reward {base_rewards[agent_id]} -> {shaped[agent_id]:.4f}  (expected {expected:.4f})"
    )

## 5.4 Ecology versus incentive

Ecological restoration belongs in the transition:

$$
I_t
=
\rho
\sum_i e_{i,t},
$$

where $\rho$ is restoration effectiveness (`ecology_cfg["restoration_effectiveness"]`).

Then:

$$
B_{t+1}
=
B_t+G(B_t)+\epsilon_t+I_t-H_t.
$$

The subsidy changes reward:

$$
r_{i,t}
\mapsto
r_{i,t}-ce_{i,t}^2+\sigma_\theta e_{i,t}.
$$

These are separate mechanisms in the scientific model, and separate objects in
the code: $\rho$ is a benchmark parameter, $\sigma_\theta$ a mechanism
parameter.

# 6. Social influence

Social influence can affect what agents observe and, in the full formulation,
can add an intrinsic reward for influencing peer behavior.

## 6.1 Full social-influence objective

For agent $i$:

$$
c_t^i
=
\sum_{j\ne i}
D_{KL}
\left[
\pi_j(a_t^j\mid a_t^i,s_t^j)
\|
\pi_j(a_t^j\mid s_t^j)
\right].
$$

A social-influence reward is:

$$
r_t^i=r_{i,t}+\beta c_t^i.
$$

$\beta$ controls influence strength.

## 6.2 What the supplied implementation does

`SocialInfluenceMechanism` implements observation augmentation only:

$$
o_{i,t}^{*}
=
[
o_{i,t},
a_{1,t-1},
\dots,
a_{j,t-1},
\dots
],
\qquad j\ne i.
$$

Each agent observes the previous-step actions of the other agents, in the order
of `agent_ids` with itself excluded. This is an observation-shaping mechanism
$\mathcal M_\theta^O$ with no optimized parameter (`dimension == 0`).

**The counterfactual KL reward bonus of Jaques et al. (2019) is not
implemented.** The constructor argument `influence_weight` ($\beta$) is
reserved for it and currently has no effect on any channel.

## 6.3 Social-influence constructor

```text
influence_weight
    beta; reserved, unused until the reward bonus is implemented.

bindings["previous_actions"]
    env -> {agent_id: previous delivered action vector}.

bindings["agent_ids"]
    env -> stable ordered sequence of participating agents.
```

In [ ]:
from core.mechanism.algorithms.social_influence import SocialInfluenceMechanism

social = SocialInfluenceMechanism(
    bindings={
        "previous_actions": lambda env: env.previous_actions,
        "agent_ids": lambda env: tuple(env.agents),
    },
)
print("dimension:", social.dimension, "| param_names:", social.param_names())

# By hand: the environment would resolve both bindings for us.
augmented = social.observation(
    base_obs, previous_actions=delivered, agent_ids=("u0", "u1")
)
for agent_id, obs in augmented.items():
    print(agent_id, obs, "<- [own obs, peer's previous action]")

## 6.4 Observation dimension

With:

$$
N=\text{number of agents},
\qquad
d_a=\text{action dimension},
$$

peer-action augmentation adds:

$$
(N-1)d_a
$$

features.

The declared Gymnasium observation space must include this expansion (see
`examples.bilevel_fishery.debug.observation_dim` in section 11).

# 7. Optional threshold penalty

The project also contains a generic smooth reward penalty below a resource
threshold, `ThresholdPenaltyMechanism`. It is a fixed rule (`dimension == 0`),
so the outer optimizer never touches it, but `to_vector()` still exposes
$(\tau, \lambda)$ to the agents.

Let:

- $b_t$: normalized resource level;
- $\tau$: threshold;
- $\lambda$: maximum penalty;
- $w$: transition width.

Then:

$$
p_t
=
\frac{\lambda}
{1+\exp((b_t-\tau)/w)},
$$

and:

$$
r_{i,t}^{*}=r_{i,t}-p_t.
$$

In [ ]:
from core.mechanism.algorithms.penalty import ThresholdPenaltyMechanism

penalty = ThresholdPenaltyMechanism(
    threshold=0.20,
    penalty_amount=0.10,
    transition_width=0.03,
    bindings={
        "resource_level": lambda env: env.S_t["fish"] / max(env.K, EPS),
    },
)
print("dimension:", penalty.dimension, "| to_vector():", penalty.to_vector())

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(levels, [penalty.penalty(b) for b in levels])
ax.axvline(penalty.threshold, ls="--", c="grey", lw=0.8)
ax.set_xlabel("normalized resource level  b")
ax.set_ylabel("penalty  p")
ax.set_title("Threshold penalty")
plt.show()

for b in (0.1, 0.2, 0.3):
    print(
        f"b={b}: rewards {base_rewards} -> {penalty.reward(base_rewards, resource_level=b)}"
    )

# 8. Stacking mechanisms

Composition lets several regulatory algorithms behave as one mechanism.

There are two semantics:

1. chained/sequential;
2. parallel + merge.

Both composites are themselves `Mechanism`s, so they can be nested and handed
to the environment or the optimizer exactly like a single algorithm.

## 8.1 Chained composition

For:

$$
(M_1,M_2,\dots,M_k),
$$

chained action composition is:

$$
M_{\text{chain}}^A(x)
=
M_k^A(
\dots
M_2^A(
M_1^A(x)
)
\dots
).
$$

The same ordering is used independently for reward and observation.

Therefore order matters when two children alter the same channel.

### Example

```python
children=(quota, subsidy, social)
```

Action channel:

```text
Quota -> Subsidy(identity) -> Social(identity)
```

Reward channel:

```text
Quota(identity) -> Subsidy -> Social(identity)
```

Observation channel:

```text
Quota -> Subsidy(identity) -> Social
```

The final observation therefore includes quota information followed by
peer-action information.

## 8.2 Optimizer representation

If child dimensions are:

$$
d_1,\dots,d_k,
$$

then:

$$
d_{\text{chain}}=\sum_i d_i.
$$

`encode()` concatenates child vectors and `decode()` slices them back into
the children. `param_names()` prefixes each name with the child's position and
class so the outer optimizer's logs stay readable.

In [ ]:
from core.mechanism.composition.chained_mechanism import ChainedMechanism

mechanism = ChainedMechanism(children=(quota, subsidy, social))

print("dimension  :", mechanism.dimension)
print("param_names:", mechanism.param_names())
print("encode()   :", mechanism.encode())
print("to_vector():", mechanism.to_vector())

# decode() propagates to the children; the result is a new ChainedMechanism.
candidate = mechanism.decode(np.array([0.30, 0.80]))
print("decoded children:")
for child in candidate.children:
    print("   ", child)
assert np.allclose(candidate.encode(), [0.30, 0.80])

The composite's channel methods take the environment as `env=` and resolve
**each child's own bindings** against it. To exercise them outside a real
environment, any object exposing the attributes the bindings read is enough.

In [ ]:
from types import SimpleNamespace

stub_env = SimpleNamespace(
    S_t={"fish": 2500.0},
    K=5000.0,
    agents=["u0", "u1"],
    previous_actions={
        "u0": np.zeros(2, dtype=np.float32),
        "u1": np.zeros(2, dtype=np.float32),
    },
)

a_star = mechanism.action(requested, env=stub_env)
r_star = mechanism.reward(base_rewards, env=stub_env, action_after=a_star)
o_star = mechanism.observation(base_obs, env=stub_env)

print("resolved bindings (quota) :", quota.resolve(stub_env))
print("resolved bindings (social):", social.resolve(stub_env))
print("a*:", a_star)
print("r*:", r_star)
print("o*:", o_star)

## 8.3 Parallel composition

In parallel composition every child receives the same original input:

$$
x
\rightarrow
\{
M_1(x),
M_2(x),
\dots,
M_k(x)
\}.
$$

A merge operator combines outputs:

$$
M_{\parallel}(x)
=
\Gamma
\left(
x,
M_1(x),
\dots,
M_k(x)
\right).
$$

Separate merge operators may be used for action, reward, and observation. Each
merge has signature `merge(original, outputs) -> merged`, where `outputs` is a
tuple following child order.

In [ ]:
from core.mechanism.composition.parallel_mechanism import ParallelMechanism


def additive_reward_merge(original, outputs):
    # Add each child's reward delta to the same base reward.
    merged = {}
    for agent_id, base_reward in original.items():
        delta = sum(float(output[agent_id]) - float(base_reward) for output in outputs)
        merged[agent_id] = float(base_reward) + delta
    return merged


def keep_original(original, outputs):
    # Identity merge for channels the children do not use.
    return original


# Subsidy and penalty both act on the reward channel and are independent:
# summing their deltas is order-independent.
incentives = ParallelMechanism(
    children=(subsidy, penalty),
    action_merge=keep_original,
    reward_merge=additive_reward_merge,
    observation_merge=keep_original,
)
print("dimension  :", incentives.dimension, "| param_names:", incentives.param_names())

low_stock_env = SimpleNamespace(S_t={"fish": 500.0}, K=5000.0)  # b = 0.10 < tau
r_parallel = incentives.reward(base_rewards, env=low_stock_env, action_after=a_star)
r_seq = penalty.reward(
    subsidy.reward(base_rewards, action_after=a_star), resource_level=0.10
)
print("parallel :", r_parallel)
print("sequential:", r_seq)
assert all(np.isclose(r_parallel[a], r_seq[a]) for a in base_rewards)

Parallel composition is only order-independent if the merge function itself
is order-independent. For additive reward deltas this holds, which is why the
parallel and chained results above coincide.

# 9. Building and stepping the fishery mechanism stack

`examples/bilevel_fishery/debug.py::build_mechanism` builds the reference
stack used by the experiment: quota on harvest, subsidy on restoration, peers'
previous actions observed.

To step the regulated environment we need the same infrastructure the
optimizers use: a `World` Ray actor acting as a blackboard. The outer optimizer
normally publishes a candidate mechanism there; the environment fetches it at
`reset()` by `mechanism_id` and `policy_seed`. Here we publish one by hand.

`RayRuntime.ensure_initialized` starts Ray in local mode and works around the
`uv run` / runtime-env quirk, so use it rather than a bare `ray.init()`.

In [ ]:
import ray
from gymnasium import spaces

from core.adaptors.ray.runtime import RayRuntime, RayRuntimeConfig
from core.world.base import World
from core.world.context import Context, MechanismContext, MechanismStatus
from examples.bilevel_fishery.debug import build_mechanism
from examples.bilevel_fishery.regulated_env import FisheryRegulatedEnv

RayRuntime.ensure_initialized(RayRuntimeConfig(num_cpus=2))
world = World.options(name="tutorial_world").remote()

AGENTS = ["u0", "u1"]
POLICY_SEED = 123
HORIZON = 20

fishery_stack = build_mechanism(social=True)
print(fishery_stack.param_names())


def publish(mech, *, index: int, seed: int) -> None:
    # Publish a candidate mechanism the way the outer optimizer would.
    ray.get(
        world.append_context.remote(
            Context(
                id=None,
                opt_id="tutorial",
                step=0,
                env="tutorial",
                payload=MechanismContext(
                    index=index,
                    env_id=None,
                    seed=seed,
                    status=MechanismStatus.published,
                    mechanism=mech,
                    metrics=None,
                ),
            )
        )
    )


publish(fishery_stack, index=0, seed=POLICY_SEED)

The environment is constructed with the *template* mechanism (it fixes the
observation size through `to_vector()`), the `mechanism_id` to fetch, and the
action spaces. Raw policy outputs are unbounded: `MultiAgentRegulatedEnv`
squashes them to $[0,1]$ with a sigmoid before any hook or mechanism runs.

In [ ]:
def make_env(
    *, mechanism_id: int, policy_seed: int, seed: int = 7
) -> FisheryRegulatedEnv:
    return FisheryRegulatedEnv(
        world=world,
        opt_id="tutorial_env",
        mechanism_id=mechanism_id,
        agents=AGENTS,
        mechanism=fishery_stack,
        horizon=HORIZON,
        seed=seed,
        policy_seed=policy_seed,
        ecology_cfg={
            "r": 0.3,
            "K": 5000.0,
            "p": 1.0,
            "fish_init": 4000.0,
            "sigma": 0.02,
            "initial_stock_log_sigma": 0.05,
            "unregulated_f_multiplier": 2.0,
            "restoration_effectiveness": 20.0,
        },
        action_spaces={
            aid: spaces.Box(-np.inf, np.inf, (2,), np.float32) for aid in AGENTS
        },
    )


env = make_env(mechanism_id=0, policy_seed=POLICY_SEED)
obs, infos = env.reset()

print("published mechanism assigned:", env.published_mechanism_assigned)
print(
    "observation layout: base",
    env.obs_map,
    "+ theta (normalized)",
    fishery_stack.to_vector(),
    "+ [allowed_frac] + peers' previous actions",
)
print(
    "theta holds fixed_quota as is and the subsidy divided by MAX_SUBSIDY:",
    f"{subsidy.subsidy} / {MAX_SUBSIDY} = {subsidy.subsidy / MAX_SUBSIDY}",
)
for agent_id, o in obs.items():
    print(agent_id, o, "shape", o.shape)

In [ ]:
# Index of the quota's allowed fraction in the final observation, derived from
# the layout rather than hard-coded: benchmark observation (env.obs_map), then
# theta = to_vector(), then the features appended by the quota's observation
# channel (the quota is the first child of the chain that touches observations).
quota_child = next(ch for ch in fishery_stack.children if isinstance(ch, QuotaMechanism))
ALLOWED_FRAC_INDEX = (
    len(env.obs_map)
    + fishery_stack.to_vector().shape[0]
    + quota_child.observation_names().index("effective_quota")
)
print("allowed_frac index in the observation:", ALLOWED_FRAC_INDEX)

rng = np.random.default_rng(0)
fish, allowed, rewards_log = [env.S_t["fish"]], [], []

for t in range(HORIZON):
    raw_actions = {aid: rng.normal(size=2).astype(np.float32) for aid in AGENTS}
    obs, rewards, terminated, truncated, infos = env.step(raw_actions)
    fish.append(env.S_t["fish"])
    allowed.append(obs["u0"][ALLOWED_FRAC_INDEX])
    rewards_log.append(rewards)

print("truncated at horizon:", truncated["__all__"])
print("last rewards:", rewards)
print("info keys:", sorted(infos["u0"]))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(np.array(fish) / env.K)
axes[0].set_title("normalized biomass  B / K")
axes[0].set_xlabel("step")
axes[1].plot(allowed)
axes[1].set_title("quota allowed fraction  alpha")
axes[1].set_xlabel("step")
plt.tight_layout()
plt.show()

The step lifecycle that produced these numbers is fixed by
`MultiAgentRegulatedEnv.step`:

```text
policy output
    -> normalize action               (sigmoid, action_temperature)
    -> benchmark @action hook          (optional; the fishery has none)
    -> mechanism.action                M^A   (quota caps harvest)
    -> benchmark @reward hook          intrinsic reward (delivered harvest fraction)
    -> benchmark @transition hook      Pella-Tomlinson dynamics
    -> mechanism.reward                M^R   (subsidy on restoration effort)
    -> benchmark @observation hook     [fish_norm, total_usage_norm]
    -> append mechanism.to_vector()    theta = [fixed_quota, subsidy / MAX_SUBSIDY]
    -> mechanism.observation           M^O   (allowed_frac, peers' previous actions)
    -> publish EnvStepContext          to the World
```

Note that `to_vector()` exposes the *encoded* parameters, not the raw ones:
`fixed_quota` is already in $[0,1]$ and is passed through unchanged, but the
subsidy rate is divided by `MAX_SUBSIDY`, so the agents see `0.2` for a
subsidy of `0.10`. The parameter names printed by `param_names()` are labels
for these normalized entries.

Every step is also recorded on the `World` as an `EnvStepContext`, which is
how the reporter and the outer optimizer read back trajectories.

In [ ]:
ctx_ids = ray.get(world.get_opt_ctx_ids.remote("tutorial_env"))
first = ray.get(world.get_context.remote(ctx_ids[0])).payload
print(f"{len(ctx_ids)} EnvStepContext records; first one:")
print("  mechanism id :", first.mechanism, "| policy_seed:", first.policy_seed)
print("  action       :", first.action)
print("  reward       :", first.reward)

# 10. Start with a quota-only smoke run

Before the full stack, validate the bilevel lifecycle with a tiny run. The
experiment script exposes everything as CLI flags; `--no-social` drops the
social-observation child, leaving quota + subsidy (two optimized parameters).

```bash
WANDB_MODE=offline uv run python -m examples.bilevel_fishery.debug \
    --outer-iters 2 --train-iters 2 --num-agents 2 --horizon 20 \
    --num-candidates 2 --num-eval-seeds 1 --num-cpus 2 --no-social
```

```text
outer iterations = 2
candidates        = 2
eval seeds        = 1
horizon           = 20
agents            = 2
```

This is the same configuration as
`tests/integration/test_fishery_mechanisms.py::test_stage_c_bilevel_smoke`.
It trains real APPO policies through Ray, which is why it is launched from the
command line rather than executed in this notebook, but it is short: a
two-generation smoke run of this kind was measured at about 30 s, so it
completes well under a minute on a laptop. Drop the flags to run the full
experiment.

# 11. Bilevel configuration anatomy

`BilevelConfig` is a fluent builder: every method below returns `self`.

```text
BilevelConfig
├── world                <- .world(world_name=...)
├── reporting            <- .reporting(reporter="wandb", project_name=..., config=None, settings_dict=None)
├── mechanism            <- .mechanism(mechanism=<Mechanism>)
├── training             <- .training(outer_iters=..., output_dir=None)
├── ray                  <- .ray(device="cpu", num_cpus=..., omp_threads=1, logging_level="ERROR", runtime_env=None)
├── outer optimizer: ES  <- .outer(ESConfig()...); dimension taken from mechanism.dimension
└── inner optimizer: APPO/RLlib  <- .inner(APPOptimizerConfig()...)
    ├── environment      <- mechanism template injected into env_config
    ├── env runners
    ├── learners
    ├── training
    ├── evaluation
    └── agents           <- observation space must match observation_dim()
```

`BilevelConfig().mechanism(mechanism=...)` is the only builder entry point for
the mechanism: the same object fixes the outer search space and serves as the
environments' default until a candidate is published.

`.reporting(...)` selects the reporting backend; only Weights & Biases is
implemented, and `reporter="local"` raises `TypeError`. Do not confuse it with
the read-only `reporter` property, which holds the `WandbReporter` actor handle
once `build_optimizer()` has created it and stays `None` before that.


## 11.1 Inspecting the reference configuration

`examples.bilevel_fishery.debug.build_config` assembles the complete, working
configuration from parsed CLI arguments. Building the configuration has no
side effects (no Ray, no W&B); only `build_optimizer()` / `run()` would start
training, so we stop before that.

In [ ]:
from examples.bilevel_fishery.debug import build_config, observation_dim, parse_args

args = parse_args(
    [
        "--outer-iters",
        "2",
        "--train-iters",
        "2",
        "--num-agents",
        "2",
        "--horizon",
        "20",
        "--num-candidates",
        "2",
        "--num-eval-seeds",
        "1",
    ]
)
cfg = build_config(args)

template = cfg.mechanism_template
print("outer search dimension :", template.dimension)
print("optimized parameters   :", template.param_names())
print("theta shown to agents  :", template.to_vector(), "(normalized: subsidy / MAX_SUBSIDY)")
print("observation dim (N=2)  :", observation_dim(template, args.num_agents))
print("observation dim (N=10) :", observation_dim(template, 10))
print("outer iterations       :", cfg.outer_iters)
print("world name             :", cfg.world_name)
# cfg.build_optimizer().run() would start the ES x APPO loop -- see section 10.

# 12. One complete step with the mechanism stack

Assume:

```text
Quota -> Subsidy -> SocialInfluence
```

and:

$$
a_{i,t}=[h_{i,t},e_{i,t}].
$$

## Action phase

Quota:

$$
[h_{i,t},e_{i,t}]
\mapsto
[h_{i,t}^{*},e_{i,t}].
$$

## Transition phase

Harvest:

$$
H_t
=
\sum_i h_{i,t}^{*}H_{i,t}^{\max}.
$$

Restoration:

$$
I_t
=
\rho\sum_i e_{i,t}.
$$

Dynamics:

$$
B_{t+1}
=
B_t+G(B_t)+\epsilon_t+I_t-H_t.
$$

## Reward phase

Benchmark computes $r_{i,t}$ (the delivered harvest fraction) **before** the
transition, on the current state and the delivered actions.

Subsidy gives:

$$
r_{i,t}^{*}
=
r_{i,t}
-
ce_{i,t}^2
+
\sigma_\theta e_{i,t}.
$$

## Observation phase

Base observation is built from the new benchmark state
$[b_{t+1}, H_t/K]$.

The environment appends $\theta$ = `mechanism.to_vector()`, i.e. the
normalized parameters `[fixed_quota, subsidy / MAX_SUBSIDY]` rather than the
raw subsidy rate.

Quota appends $\alpha_t$.

Social influence appends peer previous actions.

The learner receives the final transformed observation.

# 13. Reproducibility strategy

Before comparing training curves across mechanism variants:

1. fix all seeds (environment `seed`, `policy_seed`, RLlib seeds);
2. set stochastic growth noise to zero (`sigma`, `initial_stock_log_sigma`);
3. feed the same manual action sequence to both variants;
4. compare quota `allowed_frac`;
5. compare delivered harvest;
6. compare state transition;
7. compare reward;
8. then re-enable stochasticity and RL.

Action-transform parity is much easier to diagnose than full stochastic RL
parity. The cell below runs steps 1 to 7 for two fresh environments with the
same seed and the same published mechanism: the trajectories must coincide
exactly.

Note that `World.get_mechanism_by_id` hands a published candidate to the first
environment that asks for a given `(mechanism_id, policy_seed)` and marks it as
in training, so a second environment needs its own published copy.

In [ ]:
publish(fishery_stack, index=1, seed=201)
publish(fishery_stack, index=1, seed=202)

env_a = make_env(mechanism_id=1, policy_seed=201, seed=11)
env_b = make_env(mechanism_id=1, policy_seed=202, seed=11)
for e in (env_a, env_b):
    e.sigma = 0.0
    e.initial_stock_log_sigma = 0.0

obs_a, _ = env_a.reset()
obs_b, _ = env_b.reset()
assert env_a.published_mechanism_assigned and env_b.published_mechanism_assigned

rng = np.random.default_rng(42)
for _ in range(HORIZON):
    raw = {aid: rng.normal(size=2).astype(np.float32) for aid in AGENTS}
    obs_a, rew_a, _, _, info_a = env_a.step(raw)
    obs_b, rew_b, _, _, info_b = env_b.step(raw)
    for aid in AGENTS:
        assert np.array_equal(obs_a[aid], obs_b[aid])
        assert rew_a[aid] == rew_b[aid]
        assert info_a[aid]["delivered_harvest"] == info_b[aid]["delivered_harvest"]
    assert env_a.S_t == env_b.S_t

print("deterministic parity over", HORIZON, "steps: OK")
print("final biomass:", env_a.S_t["fish"], "| final allowed_frac:", obs_a["u0"][ALLOWED_FRAC_INDEX])

In [ ]:
ray.shutdown()

# 14. References represented in the project slides

- Madani & Dinar: exogenous regulatory institutions for common-pool resource
  management.
- Yang et al. (2022): adaptive incentive design with multi-agent
  meta-gradient reinforcement learning.
- Jaques et al. (2019): social influence as intrinsic motivation for
  multi-agent deep reinforcement learning. PMLR 97:3040-3049.
- Pella & Tomlinson (1969): a generalized stock production model.
  Inter-American Tropical Tuna Commission Bulletin, 13(3), 416-497.

The code uses these ideas as mechanism-design patterns. The implemented
algorithm may be simpler than the full paper method; most importantly, the
social mechanism only exposes peers' previous actions and does not compute the
counterfactual KL reward.

# 15. Summary

This notebook showed that a mechanism is a triple of optional transforms on the
action, reward and observation channels, plus an optimizer face (`encode`,
`decode`, `dimension`) that turns its free parameters into a point in
$[0,1]^d$. The quota acts on actions and exposes its allowance through the
observation channel, the subsidy and the threshold penalty act on rewards, and
the social-influence mechanism is observation-only: it appends the peers'
previous actions and implements none of the counterfactual reward bonus of
Jaques et al. (2019). Composition (`ChainedMechanism`, `ParallelMechanism`)
makes several algorithms behave as one mechanism whose vector is the
concatenation of its children's vectors, and bindings inject benchmark values
so that the same algorithm runs on fish, water or any other normalized
resource. We stepped the real regulated fishery against a `World` actor,
checked the observation layout (benchmark features, then the normalized
`to_vector()`, then the quota allowance and the peers' actions) and verified
deterministic parity between two seeded environments. To build your own
benchmark and mechanism on top of this API, continue with
`custom_benchmark_creation.ipynb`; the metrics and reporting stack is covered by
`visualization.ipynb` on the logging and integration branches.